# Data Cleaning and Validation

## Objectives

This notebook:

- converts date columns into datetime values;
- standardises column names;
- handles missing values according to documented rules;
- validates duplicated records and data ranges;
- preserves historically meaningful observations;
- exports cleaned Version 1 datasets;
- verifies that exported files can be reloaded successfully.

The raw files are never modified.

## Cleaning Strategy

The following decisions are applied:

1. Column names are converted to descriptive snake_case names.
2. Temperature column names include `_c` to identify degrees Celsius.
3. Dates are converted from text to pandas datetime values.
4. No temperature values are artificially imputed.
5. Twelve global records without land-average temperature measurements are removed because the project's primary global measurement is unavailable.
6. Missing maximum, minimum, and land-and-ocean values before 1850 are retained as structurally unavailable data. Removing them would discard approximately 100   years of usable land-temperature history.
7. Country records without an average temperature are removed because they cannot contribute to temperature analysis.
8. Extreme temperatures are not automatically classified as errors because geographically and historically extreme observations may be valid.
9. Raw files remain unchanged in `data/raw/v1/`. Cleaned files are exported to `data/processed/v1/`.

In [2]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "jupyter_notebooks"
    else Path.cwd()
)

RAW_FOLDER = PROJECT_ROOT / "data" / "raw" / "v1"
PROCESSED_FOLDER = PROJECT_ROOT / "data" / "processed" / "v1"

PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

GLOBAL_INPUT = RAW_FOLDER / "GlobalTemperatures.csv"
COUNTRY_INPUT = RAW_FOLDER / "GlobalLandTemperaturesByCountry.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder: {PROCESSED_FOLDER}")

Project root: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project3/CI-Project3-ClimateLens-Understanding-Global-Temperature-Change
Processed data folder: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project3/CI-Project3-ClimateLens-Understanding-Global-Temperature-Change/data/processed/v1


In [3]:
global_raw = pd.read_csv(GLOBAL_INPUT)
country_raw = pd.read_csv(COUNTRY_INPUT)

global_data = global_raw.copy()
country_data = country_raw.copy()

print(f"Global raw shape: {global_raw.shape}")
print(f"Country raw shape: {country_raw.shape}")

Global raw shape: (3192, 9)
Country raw shape: (577462, 4)


In [4]:
global_column_names = {
    "dt": "date",
    "LandAverageTemperature": "land_average_temperature_c",
    "LandAverageTemperatureUncertainty":
        "land_average_temperature_uncertainty_c",
    "LandMaxTemperature": "land_max_temperature_c",
    "LandMaxTemperatureUncertainty":
        "land_max_temperature_uncertainty_c",
    "LandMinTemperature": "land_min_temperature_c",
    "LandMinTemperatureUncertainty":
        "land_min_temperature_uncertainty_c",
    "LandAndOceanAverageTemperature":
        "land_ocean_average_temperature_c",
    "LandAndOceanAverageTemperatureUncertainty":
        "land_ocean_average_temperature_uncertainty_c",
}

country_column_names = {
    "dt": "date",
    "AverageTemperature": "average_temperature_c",
    "AverageTemperatureUncertainty":
        "average_temperature_uncertainty_c",
    "Country": "country",
}

global_data = global_data.rename(columns=global_column_names)
country_data = country_data.rename(columns=country_column_names)

print("Global columns:")
print(global_data.columns.tolist())

print("\nCountry columns:")
print(country_data.columns.tolist())

Global columns:
['date', 'land_average_temperature_c', 'land_average_temperature_uncertainty_c', 'land_max_temperature_c', 'land_max_temperature_uncertainty_c', 'land_min_temperature_c', 'land_min_temperature_uncertainty_c', 'land_ocean_average_temperature_c', 'land_ocean_average_temperature_uncertainty_c']

Country columns:
['date', 'average_temperature_c', 'average_temperature_uncertainty_c', 'country']


In [5]:
# Convert date strings into datetime values.
global_data["date"] = pd.to_datetime(
    global_data["date"],
    format="%Y-%m-%d",
    errors="coerce",
)

country_data["date"] = pd.to_datetime(
    country_data["date"],
    format="%Y-%m-%d",
    errors="coerce",
)

# Confirm that no dates became invalid during conversion.
assert global_data["date"].notna().all(), (
    "Invalid dates were found in the global dataset."
)

assert country_data["date"].notna().all(), (
    "Invalid dates were found in the country dataset."
)

# Remove accidental spaces around geographical labels.
country_data["country"] = country_data["country"].str.strip()

assert country_data["country"].notna().all()
assert country_data["country"].ne("").all()

print("All dates and country labels passed validation.")

All dates and country labels passed validation.


In [6]:
# Count records before cleaning.
global_rows_before = len(global_data)
country_rows_before = len(country_data)

# Remove global rows where the main land-average measurement or its uncertainty is unavailable.
global_clean = global_data.dropna(
    subset=[
        "land_average_temperature_c",
        "land_average_temperature_uncertainty_c",
    ]
).copy()

# Remove country records without a usable temperature measurement.
country_clean = country_data.dropna(
    subset=["average_temperature_c"]
).copy()

# Add convenient time fields for later analysis and dashboard filtering.
for dataframe in [global_clean, country_clean]:
    dataframe["year"] = dataframe["date"].dt.year
    dataframe["month"] = dataframe["date"].dt.month

# Sort records into a consistent chronological order.
global_clean = (
    global_clean
    .sort_values("date")
    .reset_index(drop=True)
)

country_clean = (
    country_clean
    .sort_values(["country", "date"])
    .reset_index(drop=True)
)

global_rows_removed = global_rows_before - len(global_clean)
country_rows_removed = country_rows_before - len(country_clean)

print(f"Global records removed: {global_rows_removed:,}")
print(f"Country records removed: {country_rows_removed:,}")

Global records removed: 12
Country records removed: 32,651


In [7]:
global_column_order = [
    "date",
    "year",
    "month",
    "land_average_temperature_c",
    "land_average_temperature_uncertainty_c",
    "land_max_temperature_c",
    "land_max_temperature_uncertainty_c",
    "land_min_temperature_c",
    "land_min_temperature_uncertainty_c",
    "land_ocean_average_temperature_c",
    "land_ocean_average_temperature_uncertainty_c",
]

country_column_order = [
    "date",
    "year",
    "month",
    "country",
    "average_temperature_c",
    "average_temperature_uncertainty_c",
]

global_clean = global_clean[global_column_order]
country_clean = country_clean[country_column_order]

display(global_clean.head())
display(country_clean.head())

,date,year,month,land_average_temperature_c,land_average_temperature_uncertainty_c,land_max_temperature_c,land_max_temperature_uncertainty_c,land_min_temperature_c,land_min_temperature_uncertainty_c,land_ocean_average_temperature_c,land_ocean_average_temperature_uncertainty_c
0,1750-01-01,1750,1,3.034,3.574,NaN,NaN,NaN,NaN,NaN,NaN
1,1750-02-01,1750,2,3.083,3.702,NaN,NaN,NaN,NaN,NaN,NaN
2,1750-03-01,1750,3,5.626,3.076,NaN,NaN,NaN,NaN,NaN,NaN
3,1750-04-01,1750,4,8.490,2.451,NaN,NaN,NaN,NaN,NaN,NaN
4,1750-05-01,1750,5,11.573,2.072,NaN,NaN,NaN,NaN,NaN,NaN


,date,year,month,country,average_temperature_c,average_temperature_uncertainty_c
0,1838-04-01,1838,4,Afghanistan,13.008,2.586
1,1838-06-01,1838,6,Afghanistan,23.950,2.510
2,1838-07-01,1838,7,Afghanistan,26.877,2.883
3,1838-08-01,1838,8,Afghanistan,24.938,2.992
4,1838-09-01,1838,9,Afghanistan,18.981,2.538


In [8]:
# Validate the expected number of records for this dataset version.
assert len(global_clean) == 3180, (
    f"Unexpected global row count: {len(global_clean)}"
)

assert len(country_clean) == 544811, (
    f"Unexpected country row count: {len(country_clean)}"
)

# Confirm that unique record identifiers remain unique.
assert not global_clean.duplicated(subset=["date"]).any()

assert not country_clean.duplicated(
    subset=["country", "date"]
).any()

# The cleaned country dataset should contain complete temperature and uncertainty measurements.
assert country_clean["average_temperature_c"].notna().all()

assert (
    country_clean["average_temperature_uncertainty_c"]
    .notna()
    .all()
)

# Uncertainty cannot logically be negative.
global_uncertainty_columns = [
    column
    for column in global_clean.columns
    if "uncertainty" in column
]

assert not (
    global_clean[global_uncertainty_columns] < 0
).any().any()

assert not (
    country_clean["average_temperature_uncertainty_c"] < 0
).any()

print("All cleaned-data validation checks passed.")

All cleaned-data validation checks passed.


In [9]:
cleaning_summary = pd.DataFrame({
    "dataset": ["global_temperatures", "country_temperatures"],
    "raw_rows": [len(global_raw), len(country_raw)],
    "removed_rows": [global_rows_removed, country_rows_removed],
    "cleaned_rows": [len(global_clean), len(country_clean)],
    "cleaned_columns": [
        global_clean.shape[1],
        country_clean.shape[1],
    ],
    "remaining_missing_cells": [
        int(global_clean.isna().sum().sum()),
        int(country_clean.isna().sum().sum()),
    ],
})

display(cleaning_summary)

raw_country_labels = set(country_data["country"].unique())
clean_country_labels = set(country_clean["country"].unique())

excluded_country_labels = sorted(
    raw_country_labels - clean_country_labels
)

print(
    "Distinct labels before cleaning:",
    country_data["country"].nunique(),
)

print(
    "Distinct labels after cleaning:",
    country_clean["country"].nunique(),
)

print("Labels excluded from analysis:", excluded_country_labels)

,dataset,raw_rows,removed_rows,cleaned_rows,cleaned_columns,remaining_missing_cells
0,global_temperatures,3192,12,3180,11,7128
1,country_temperatures,577462,32651,544811,6,0


Distinct labels before cleaning: 243
Distinct labels after cleaning: 242
Labels excluded from analysis: ['Antarctica']


In [10]:
def missing_value_summary(dataframe):
    """Return missing-value counts and percentages by column."""
    summary = pd.DataFrame({
        "missing_count": dataframe.isna().sum(),
        "missing_percentage": (
            dataframe.isna().mean() * 100
        ).round(2),
    })

    return summary.sort_values(
        "missing_percentage",
        ascending=False,
    )


print("Global cleaned data:")
display(missing_value_summary(global_clean))

print("Country cleaned data:")
display(missing_value_summary(country_clean))

Global cleaned data:


,missing_count,missing_percentage
land_max_temperature_c,1188,37.36
land_max_temperature_uncertainty_c,1188,37.36
land_min_temperature_c,1188,37.36
land_min_temperature_uncertainty_c,1188,37.36
land_ocean_average_temperature_c,1188,37.36
land_ocean_average_temperature_uncertainty_c,1188,37.36
date,0,0.00
year,0,0.00
month,0,0.00
land_average_temperature_c,0,0.00


Country cleaned data:


,missing_count,missing_percentage
date,0,0.0
year,0,0.0
month,0,0.0
country,0,0.0
average_temperature_c,0,0.0
average_temperature_uncertainty_c,0,0.0


In [11]:
global_temperature_columns = [
    column
    for column in global_clean.columns
    if "temperature_c" in column
    and "uncertainty" not in column
]

country_temperature_columns = [
    "average_temperature_c",
    "average_temperature_uncertainty_c",
]

print("Global numerical summary:")
display(
    global_clean[global_temperature_columns]
    .describe()
    .transpose()
)

print("Country numerical summary:")
display(
    country_clean[country_temperature_columns]
    .describe()
    .transpose()
)

Global numerical summary:


,count,mean,std,min,25%,50%,75%,max
land_average_temperature_c,3180.0,8.374731,4.381310,-2.080,4.3120,8.6105,12.54825,19.021
land_max_temperature_c,1992.0,14.350601,4.309579,5.900,10.2120,14.7600,18.45150,21.320
land_min_temperature_c,1992.0,2.743595,4.155835,-5.407,-1.3345,2.9495,6.77875,9.715
land_ocean_average_temperature_c,1992.0,15.212566,1.274093,12.475,14.0470,15.2510,16.39625,17.611


Country numerical summary:


,count,mean,std,min,25%,50%,75%,max
average_temperature_c,544811.0,17.193354,10.953966,-37.658,10.025,20.901,25.814,38.842
average_temperature_uncertainty_c,544811.0,1.019190,1.202634,0.052,0.323,0.571,1.207,15.003


In [12]:
GLOBAL_OUTPUT = (
    PROCESSED_FOLDER / "global_temperatures_clean.csv"
)

COUNTRY_OUTPUT = (
    PROCESSED_FOLDER / "country_temperatures_clean.csv"
)

SUMMARY_OUTPUT = (
    PROCESSED_FOLDER / "cleaning_summary.csv"
)

global_clean.to_csv(
    GLOBAL_OUTPUT,
    index=False,
    date_format="%Y-%m-%d",
)

country_clean.to_csv(
    COUNTRY_OUTPUT,
    index=False,
    date_format="%Y-%m-%d",
)

cleaning_summary.to_csv(
    SUMMARY_OUTPUT,
    index=False,
)

print(f"Saved: {GLOBAL_OUTPUT}")
print(f"Saved: {COUNTRY_OUTPUT}")
print(f"Saved: {SUMMARY_OUTPUT}")

Saved: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project3/CI-Project3-ClimateLens-Understanding-Global-Temperature-Change/data/processed/v1/global_temperatures_clean.csv
Saved: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project3/CI-Project3-ClimateLens-Understanding-Global-Temperature-Change/data/processed/v1/country_temperatures_clean.csv
Saved: /Users/ewa/Documents/vscode-projects/code-institute-2026/CI-Project3/CI-Project3-ClimateLens-Understanding-Global-Temperature-Change/data/processed/v1/cleaning_summary.csv


In [13]:
global_reloaded = pd.read_csv(
    GLOBAL_OUTPUT,
    parse_dates=["date"],
)

country_reloaded = pd.read_csv(
    COUNTRY_OUTPUT,
    parse_dates=["date"],
)

pd.testing.assert_frame_equal(
    global_clean,
    global_reloaded,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

pd.testing.assert_frame_equal(
    country_clean,
    country_reloaded,
    check_dtype=False,
    check_exact=False,
    rtol=1e-12,
    atol=1e-12,
)

print("Export verification passed.")
print(f"Reloaded global shape: {global_reloaded.shape}")
print(f"Reloaded country shape: {country_reloaded.shape}")

Export verification passed.
Reloaded global shape: (3180, 11)
Reloaded country shape: (544811, 6)


## Cleaning Conclusions

- The raw source files were preserved without modification.
- Dates were converted into valid datetime values.
- Columns were renamed using descriptive snake_case names.
- Twelve global records without land-average measurements were removed.
- Missing global maximum, minimum, and land-and-ocean measurements before 1850 were retained as structurally unavailable data.
- A total of 32,651 country records without average-temperature values were removed.
- The analytical country dataset contains 242 geographical labels.
- Antarctica was excluded because none of its records contained an average temperature measurement.
- No duplicated global dates or country-and-date combinations were found.
- No negative uncertainty values were found.
- No temperature values were imputed.
- The exported files were reloaded and successfully validated.

The cleaned datasets are now suitable for exploratory analysis. Individual analyses will still filter records according to the availability of the specific temperature measurement being studied.